# SH01 local AI video editing — Kaggle GPU workflow

This notebook is a clean, staged smoke-test and orchestration workflow for
the `feature/ai-compose-kaggle` server branch. It keeps the video renderer
local to the Kaggle runtime: a small local Transformers director proposes
a validated composition plan, while the server and FFmpeg perform the
bounded media work.

Run the cells from top to bottom. The long/remote tests are explicitly
opt-in so the basic workflow remains fast and predictable.


## 0. Canonical setup cells

The next five cells are the only setup content carried over from the
supplied canonical notebook: repository clone, Python/GPU package check,
NVIDIA check, dependency installation, and the private `.env` heredoc.


In [ ]:
%%bash
set -e

# Clone or update only a clean checkout.  A dirty checkout is retained because
# it may contain an uploaded renderer patch; this avoids the merge conflict
# that otherwise occurs when a Kaggle session is re-run after patching files.
REPO_DIR=/kaggle/working/sh01-fastapi-server
REPO_REF="${AI_COMPOSE_REPO_REF:-feature/ai-compose-kaggle}"
REPO_URL="${AI_COMPOSE_REPO_URL:-https://github.com/BarrieBretton/sh01-fastapi-server.git}"

if [ -d "$REPO_DIR/.git" ]; then
    if [ -n "$(git -C "$REPO_DIR" status --porcelain)" ]; then
        echo "Existing checkout is dirty; retaining it without pull: $REPO_DIR"
    else
        git -C "$REPO_DIR" fetch origin "$REPO_REF"
        git -C "$REPO_DIR" checkout "$REPO_REF"
        git -C "$REPO_DIR" pull --ff-only origin "$REPO_REF"
    fi
else
    git clone -b "$REPO_REF" "$REPO_URL" "$REPO_DIR"
fi

cd "$REPO_DIR"
if [ -f requirements.txt ]; then
    python -m pip install -q -r requirements.txt
fi
python -m pip install -q -r requirements-ai-compose.txt


In [ ]:
import torch
import bitsandbytes

print("Torch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
%cd /kaggle/working/sh01-fastapi-server

!ls -1 requirements*.txt
print("Dependencies were installed in the canonical clone cell before importing torch.")


In [ ]:
import os
os.environ["MEDIA_ALLOW_UNLIMITED_RESOURCES"] = "true"
os.environ["AI_COMPOSE_ALLOW_DETERMINISTIC_RECOVERY"]="true"

In [ ]:
%%bash

cat > .env <<'EOF'
# ... REDACTED ...
EOF

## Private `.env` contract

This notebook intentionally does not use Kaggle Secrets. The private
`.env` heredoc above is loaded into the process and passed to the local
Uvicorn subprocess. Never print the file or commit this notebook while it
contains real credentials.

Useful values include `MEDIA_CLIP_API_KEY`, `AI_COMPOSE_SOURCE_PATH`,
`AI_COMPOSE_HF_MODEL`, `AI_COMPOSE_HF_LOAD_4BIT=true`,
`MEDIA_JOB_STORE_MODE=local`, `AI_COMPOSE_WORK_DIR`, and optional local
asset/SFX directories. Pexels, Pixabay, Unsplash, and Freesound keys are
optional and are only used by the opt-in remote-asset test.


In [ ]:
from pathlib import Path
from dotenv import load_dotenv

ENV_PATH = Path("/kaggle/working/sh01-fastapi-server/.env")
if not ENV_PATH.is_file():
    raise FileNotFoundError(f"Private .env file is missing: {ENV_PATH}")
assert load_dotenv(ENV_PATH, override=True)
print("Private .env loaded from:", ENV_PATH)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 1. Runtime bootstrap and input discovery

This cell establishes safe defaults, finds the mounted source video, and
prepares the local API client. It does not expose secret values.


In [ ]:
# Keep the private .env value when present. Edit this fallback only when selecting a different mounted source.
os.environ.setdefault(
    "AI_COMPOSE_SOURCE_PATH",
    "/kaggle/input/datasets/vivanjaiswal/video1/military_seth_raw_cut1.mp4",
)

In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
import time
import uuid
import zipfile
from pathlib import Path

import requests
from dotenv import load_dotenv

REPO = Path("/kaggle/working/sh01-fastapi-server").resolve()
if not REPO.exists():
    raise FileNotFoundError(f"Repository is missing: {REPO}")
os.chdir(REPO)
load_dotenv(REPO / ".env", override=True)


# Optional: attach the complete renderer patch zip as a private Kaggle input
# and set AI_COMPOSE_PATCH_ZIP in .env.  This is useful until the v3 patch is
# pushed to the feature branch.  If the checkout already contains the patch,
# leave this empty.
PATCH_ZIP = os.environ.get("AI_COMPOSE_PATCH_ZIP", "").strip()
if PATCH_ZIP:
    patch_path = Path(PATCH_ZIP).expanduser().resolve()
    if not patch_path.is_file():
        raise FileNotFoundError(f"AI_COMPOSE_PATCH_ZIP does not exist: {patch_path}")
    with zipfile.ZipFile(patch_path) as archive:
        for member in archive.infolist():
            target = (REPO / member.filename).resolve()
            if target != REPO and REPO not in target.parents:
                raise RuntimeError(f"Unsafe patch member: {member.filename}")
            archive.extract(member, REPO)
    print("Applied private patch zip:", patch_path)

# Local Kaggle defaults. Existing .env values win where appropriate.
os.environ.setdefault("MEDIA_JOB_STORE_MODE", "local")
os.environ.setdefault("MEDIA_QUEUE_CONSUMER_ENABLED", "false")
os.environ.setdefault("AI_COMPOSE_HF_MODEL", "Qwen/Qwen2.5-3B-Instruct")
os.environ.setdefault("AI_COMPOSE_HF_LOAD_4BIT", "true")
os.environ.setdefault("AI_COMPOSE_HF_MAX_NEW_TOKENS", "1800")
os.environ.setdefault("AI_COMPOSE_ALLOW_DETERMINISTIC_RECOVERY", "true")
os.environ.setdefault("AI_COMPOSE_WORK_DIR", "/kaggle/working/ai-compose-jobs")
os.environ.setdefault("MEDIA_ALLOW_UNLIMITED_RESOURCES", "true")

API_KEY = os.environ.get("MEDIA_CLIP_API_KEY", "").strip()
if not API_KEY:
    raise RuntimeError("MEDIA_CLIP_API_KEY is missing from the private .env file")

BASE = "http://127.0.0.1:5000"
HEADERS = {"X-API-KEY": API_KEY}
WORK_ROOT = Path(os.environ["AI_COMPOSE_WORK_DIR"])
OUTPUT_ROOT = Path("/kaggle/working/ai-compose-output")
WORK_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

source_env = os.environ.get("AI_COMPOSE_SOURCE_PATH", "").strip()
if source_env:
    SOURCE = Path(source_env).expanduser().resolve()
else:
    mounted = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.exists():
            mounted.extend(
                p for p in root.rglob("*")
                if p.is_file()
                and p.suffix.lower() in {".mp4", ".webm", ".mov", ".mkv"}
                and "ai-compose-jobs" not in str(p)
                and "ai-compose-output" not in str(p)
            )
    if len(mounted) != 1:
        sample = "\n".join(str(p) for p in sorted(mounted)[:20])
        raise FileNotFoundError(
            "Set AI_COMPOSE_SOURCE_PATH in .env; automatic discovery found "
            f"{len(mounted)} candidates:\n{sample}"
        )
    SOURCE = mounted[0].resolve()

if not SOURCE.is_file():
    raise FileNotFoundError(f"Source video does not exist: {SOURCE}")

# This must be set before Uvicorn starts. Setting it after server startup is too late.
os.environ["MEDIA_LOCAL_SOURCE_DIR"] = str(SOURCE.parent)

print("Repository:", REPO)
print("Source:", SOURCE)
print("Source size MiB:", round(SOURCE.stat().st_size / 1024**2, 2))
print("HF model:", os.environ["AI_COMPOSE_HF_MODEL"])
print("Private API key loaded: yes")

## 2. Preflight: repository contract, FFmpeg, and GPU

The server must be syntactically valid before a background process is
started. The checks also confirm that the refinement fields and JSON-repair
dependency needed by the local director are present.


In [ ]:
import importlib.util
import py_compile

required_files = [
    REPO / "app.py",
    REPO / "media_dispatcher.py",
    REPO / "ai_compose.py",
    REPO / "ai_compose_engine.py",
    REPO / "ai_compose_schema.py",
    REPO / "ai_compose_refinement.py",
    REPO / "ai_compose_policy.py",
    REPO / "ai_compose_ir.py",
    REPO / "ai_compose_quality.py",
]
for path in required_files:
    if not path.is_file():
        raise FileNotFoundError(path)
    py_compile.compile(str(path), doraise=True)

import sys
sys.path.insert(0, str(REPO))
from ai_compose_schema import AIComposeRequest

assert "refinement_iterations" in AIComposeRequest.model_fields
assert "editorial_policy" in AIComposeRequest.model_fields

ffmpeg = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
if ffmpeg.returncode != 0:
    raise RuntimeError(ffmpeg.stderr)

branch = subprocess.run(
    ["git", "branch", "--show-current"], cwd=REPO,
    capture_output=True, text=True, check=True,
).stdout.strip()
commit = subprocess.run(
    ["git", "rev-parse", "--short", "HEAD"], cwd=REPO,
    capture_output=True, text=True, check=True,
).stdout.strip()
print("AI-compose branch:", branch or "detached")
print("AI-compose commit:", commit)
print("Contract preflight: PASS")
from ai_compose_schema import CompositionPlan
assert "graphics" in CompositionPlan.model_fields

from ai_compose_schema import CompositionPlan
assert "graphics" in CompositionPlan.model_fields

from ai_compose_schema import CompositionPlan
assert "graphics" in CompositionPlan.model_fields
assert importlib.util.find_spec("json_repair"), "json-repair is not installed"


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 3. Start one non-blocking local server

Uvicorn is launched with `Popen`, so this notebook remains interactive.
The subprocess inherits the private `.env` values. Queue/Redis and remote
Postgres are not required for these local smoke tests.


In [ ]:
import time

SERVER_LOG = Path("/kaggle/working/sh01_ai_compose_server.log")
if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    print("Server already running:", SERVER_PROCESS.pid)
else:
    SERVER_LOG_HANDLE = SERVER_LOG.open("w", encoding="utf-8")
    SERVER_PROCESS = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app:app", "--host", "0.0.0.0", "--port", "5000", "--workers", "1"],
        cwd=str(REPO),
        env=os.environ.copy(),
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        text=True,
    )

deadline = time.monotonic() + 180
while time.monotonic() < deadline:
    if SERVER_PROCESS.poll() is not None:
        tail = SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-8000:]
        raise RuntimeError(f"Server exited with {SERVER_PROCESS.returncode}:\n{tail}")
    try:
        probe = requests.get(f"{BASE}/openapi.json", timeout=5)
        if probe.ok:
            print("Server ready; PID:", SERVER_PROCESS.pid)
            break
    except requests.RequestException:
        pass
    time.sleep(2)
else:
    tail = SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-8000:]
    raise TimeoutError(f"Server did not become ready:\n{tail}")


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 4. Capabilities and schema contract


In [ ]:
capabilities = requests.get(f"{BASE}/media/ai-compose-capabilities", headers=HEADERS, timeout=30)
capabilities.raise_for_status()
CAPABILITIES = capabilities.json()

schema_response = requests.get(f"{BASE}/media/ai-compose-schema", headers=HEADERS, timeout=30)
schema_response.raise_for_status()
SCHEMA = schema_response.json()

assert "versioned-edit-plan" in CAPABILITIES.get("features", [])
assert "technical-quality-review" in CAPABILITIES.get("features", [])
assert "rendered-motion-graphics" in CAPABILITIES.get("features", [])
assert "animated-camera-keyframes" in CAPABILITIES.get("features", [])
assert "xfade-transitions" in CAPABILITIES.get("features", [])
assert "editorial_policy" in SCHEMA.get("request", {}).get("properties", {})
assert "edit_plan" in SCHEMA

print(json.dumps({
    "workflow": CAPABILITIES.get("workflow"),
    "features": CAPABILITIES.get("features"),
    "asset_sources": CAPABILITIES.get("asset_sources"),
    "animations": CAPABILITIES.get("animations"),
    "schema_sections": sorted(SCHEMA.keys()),
}, indent=2))

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 4A. Catalog-inspired capability coverage and honest boundaries

`awesome-claude-video-skills` is a catalog of hundreds of independent
projects, not a single dependency that can be copied wholesale.  The matrix
below maps the useful capability families from that catalog to this server's
actual implementation and to the smoke test that exercises each one.

The server is intentionally FFmpeg-first and local-first.  A future
Remotion/HyperFrames worker can consume the same `sh01/ai-edit@2` plan, but
this notebook does not label that future backend as implemented.


In [ ]:
CATALOG_CAPABILITY_MATRIX = [
    {"family": "agentic direction + bounded refinement", "status": "implemented", "test": "Smoke C"},
    {"family": "transcript-driven silence-aware cuts", "status": "implemented", "test": "Smoke A/B"},
    {"family": "typed edit program / source-to-program timeline", "status": "implemented", "test": "Smoke D2"},
    {"family": "captions and ASS export", "status": "implemented", "test": "Smoke A/B"},
    {"family": "motion graphics / hook / callout overlays", "status": "implemented", "test": "Smoke D2"},
    {"family": "animated camera keyframes / punch-ins", "status": "implemented", "test": "Smoke D2"},
    {"family": "transitions", "status": "implemented", "test": "Smoke D2"},
    {"family": "local B-roll / partial-frame layers", "status": "implemented-optional", "test": "Smoke D"},
    {"family": "remote royalty-free assets", "status": "implemented-optional", "test": "Remote asset"},
    {"family": "SFX / local TTS / music inputs", "status": "implemented-optional", "test": "Audio branches"},
    {"family": "multi-variant output", "status": "implemented", "test": "Smoke E"},
    {"family": "evidence, receipts, mechanical QC", "status": "implemented", "test": "Review gate"},
    {"family": "HTML/Remotion/HyperFrames renderer", "status": "future-backend", "test": "not run"},
    {"family": "true multitrack editable project / undo-redo", "status": "future-backend", "test": "not run"},
    {"family": "beat-synchronous edit decisions", "status": "future-backend", "test": "not run"},
    {"family": "independent visual critic / storyboard approval UI", "status": "partial", "test": "Review gate"},
    {"family": "reusable style-skill packs / template library", "status": "partial", "test": "Editorial policy matrix"},
    {"family": "avatar / lip-sync / generative scene pipelines", "status": "future-backend", "test": "not run"},
]

print(json.dumps(CATALOG_CAPABILITY_MATRIX, indent=2))
implemented = {row["family"] for row in CATALOG_CAPABILITY_MATRIX if row["status"] == "implemented"}
assert "typed edit program / source-to-program timeline" in implemented
assert "motion graphics / hook / callout overlays" in implemented
assert "animated camera keyframes / punch-ins" in implemented
assert "evidence, receipts, mechanical QC" in implemented
print("Catalog mapping: PASS (implemented, optional, partial, and future capabilities are explicitly separated)")


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the catalog coverage matrix is complete
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 4B. Generalized editorial-policy matrix

This is a no-render contract test across different niches.  It verifies that
the policy resolver is content-aware without hard-coding the podcast example:
talking head, tutorial, product, documentary, vlog, montage, and music inputs
all produce a normalized policy before direction starts.


In [ ]:
from ai_compose_policy import resolve_editorial_policy

CONTENT_PROFILES = [
    "talking_head", "podcast", "interview", "tutorial", "screen_demo",
    "product", "documentary", "vlog", "montage", "music",
]
policy_matrix = []
for content_type in CONTENT_PROFILES:
    request = AIComposeRequest.model_validate({
        "source_path": str(SOURCE),
        "brief": f"Make a strong {content_type} short for its audience.",
        "preset": "viral_short" if content_type in {"vlog", "montage", "music"} else "clean_explainer",
        "target_seconds": 15,
        "fetch_assets": False,
        "editorial_policy": {
            "content_type": content_type,
            "source_audio": "preserve",
            "voiceover": "auto",
            "supplementary_visuals": "auto",
            "motion_graphics": "auto",
            "camera_motion": "auto",
        },
    })
    resolved = resolve_editorial_policy(request, {"duration": 36.8, "has_audio": True}, [])
    policy_matrix.append({"input": content_type, **resolved.model_dump(mode="json")})

print(json.dumps(policy_matrix, indent=2))
assert {row["input"] for row in policy_matrix} == set(CONTENT_PROFILES)
print("Editorial-policy matrix: PASS")


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the generalized policy matrix is complete
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 5. Reusable job helpers

Every request receives a fresh ID. Polling uses the local status endpoint,
not the optional public/Cloudflare URL returned by the API. The helper
prints only state changes and preserves the final receipt for later review.


In [ ]:
from IPython.display import Video, display

JOB_RECEIPTS = []

def submit_job(payload: dict) -> dict:
    payload = dict(payload)
    payload.setdefault("request_id", f"kaggle-{uuid.uuid4()}")
    response = requests.post(
        f"{BASE}/media/ai-compose",
        headers={**HEADERS, "Content-Type": "application/json"},
        json=payload,
        timeout=120,
    )
    print("submit:", response.status_code)
    if not response.ok:
        print(response.text[:4000])
    response.raise_for_status()
    job = response.json()
    print("job_id:", job["job_id"])
    return job

def poll_job(job: dict, timeout_seconds: int = 3600, interval_seconds: int = 5) -> dict:
    job_id = job["job_id"]
    status_url = f"{BASE}/media/ai-compose-jobs/{job_id}"
    deadline = time.monotonic() + timeout_seconds
    previous = None
    while time.monotonic() < deadline:
        response = requests.get(status_url, headers=HEADERS, timeout=60)
        response.raise_for_status()
        status = response.json()
        marker = (
            status.get("status"), status.get("stage"), status.get("progress"),
            status.get("refinement_round"), status.get("error")
        )
        if marker != previous:
            print(
                "status=", status.get("status"),
                "stage=", status.get("stage"),
                "progress=", status.get("progress"),
                "round=", status.get("refinement_round"),
                "error=", status.get("error"),
            )
            previous = marker
        if status.get("status") in {"complete", "failed"}:
            JOB_RECEIPTS.append(status)
            if status.get("status") != "complete":
                raise RuntimeError(json.dumps(status, indent=2))
            return status
        time.sleep(interval_seconds)
    raise TimeoutError(f"Timed out waiting for job {job_id}")

def download_variant(status: dict, variant: int = 1) -> Path:
    job_id = status["job_id"]
    filename = f"variant_{variant:02d}.mp4"
    destination = Path("/kaggle/working") / f"ai-compose-{job_id}-{filename}"
    response = requests.get(
        f"{BASE}/media/ai-compose-files/{job_id}/{filename}",
        headers=HEADERS,
        timeout=900,
    )
    response.raise_for_status()
    destination.write_bytes(response.content)
    print("downloaded:", destination, "MiB:", round(destination.stat().st_size / 1024**2, 2))
    return destination

def archive_job(status: dict, label: str) -> Path:
    """Copy every locally available output and receipt into one durable folder."""
    job_id = status["job_id"]
    source_job_root = WORK_ROOT / job_id
    destination = OUTPUT_ROOT / f"{label}-{job_id}"
    destination.mkdir(parents=True, exist_ok=True)

    for output in status.get("outputs", []):
        variant = int(output.get("variant", 1))
        filename = f"variant_{variant:02d}.mp4"
        source_path = Path(output.get("local_path", ""))
        if source_path.is_file():
            shutil.copy2(source_path, destination / filename)
        else:
            response = requests.get(
                f"{BASE}/media/ai-compose-files/{job_id}/{filename}",
                headers=HEADERS, timeout=900,
            )
            response.raise_for_status()
            (destination / filename).write_bytes(response.content)

    for filename in ("composition_receipt.json", "evidence.json"):
        path = source_job_root / filename
        if path.is_file():
            shutil.copy2(path, destination / filename)

    for path in source_job_root.glob("variant_*.ass"):
        shutil.copy2(path, destination / path.name)

    (destination / "status.json").write_text(
        json.dumps(status, indent=2, ensure_ascii=False), encoding="utf-8"
    )
    print("archived:", destination)
    return destination

def show_video(path: Path) -> None:
    display(Video(str(path), embed=True, width=420))

def _probe_video(path: Path) -> dict:
    result = subprocess.run(
        [
            "ffprobe", "-v", "error", "-show_streams", "-show_format",
            "-of", "json", str(path),
        ],
        capture_output=True, text=True, check=True,
    )
    return json.loads(result.stdout)


def output_motion_score(path: Path, sample_count: int = 8) -> float:
    # Sample frames and return mean grayscale frame-to-frame change.
    from PIL import Image
    import numpy as np

    metadata = _probe_video(path)
    duration = float(metadata.get("format", {}).get("duration", 0.0) or 0.0)
    if duration <= 0.0:
        return 0.0
    preview_dir = OUTPUT_ROOT / "_motion-check" / path.stem
    preview_dir.mkdir(parents=True, exist_ok=True)
    frames = []
    for index in range(sample_count):
        timestamp = min(duration - 0.05, duration * (index + 0.5) / sample_count)
        frame_path = preview_dir / f"frame_{index:02d}.jpg"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(timestamp),
             "-i", str(path), "-frames:v", "1", "-vf", "scale=160:-1",
             str(frame_path)],
            check=True,
        )
        frames.append(np.asarray(Image.open(frame_path).convert("L"), dtype=np.float32))
    if len(frames) < 2:
        return 0.0
    return float(np.mean([np.abs(frames[i] - frames[i - 1]).mean() for i in range(1, len(frames))]) / 255.0)


def show_contact_sheet(path: Path, label: str, sample_count: int = 6) -> Path:
    # Create a human-review storyboard contact sheet from a rendered MP4.
    from PIL import Image, ImageDraw, ImageOps

    metadata = _probe_video(path)
    duration = float(metadata.get("format", {}).get("duration", 0.0) or 0.0)
    preview_dir = OUTPUT_ROOT / "_storyboards" / label.replace("/", "_")
    preview_dir.mkdir(parents=True, exist_ok=True)
    thumbs = []
    for index in range(sample_count):
        timestamp = min(max(0.0, duration - 0.05), duration * (index + 0.5) / sample_count)
        frame_path = preview_dir / f"frame_{index:02d}.jpg"
        subprocess.run(
            ["ffmpeg", "-y", "-loglevel", "error", "-ss", str(timestamp),
             "-i", str(path), "-frames:v", "1", "-vf", "scale=240:-1",
             str(frame_path)],
            check=True,
        )
        image = Image.open(frame_path).convert("RGB")
        canvas = Image.new("RGB", (260, 300), "#111111")
        canvas.paste(ImageOps.contain(image, (240, 260)), (10, 10))
        ImageDraw.Draw(canvas).text((10, 278), f"{index + 1}/{sample_count}  t={timestamp:.2f}s", fill="white")
        thumbs.append(canvas)
    columns = 3
    rows = (len(thumbs) + columns - 1) // columns
    sheet = Image.new("RGB", (columns * 260, rows * 300), "#000000")
    for index, thumb in enumerate(thumbs):
        sheet.paste(thumb, ((index % columns) * 260, (index // columns) * 300))
    output = preview_dir / "contact_sheet.jpg"
    sheet.save(output, quality=92)
    display(sheet)
    print("Storyboard contact sheet:", output)
    return output


def review_rendered_output(status: dict, video: Path, label: str, expected_duration: float) -> dict:
    # Run the notebook-level technical and human-review gate.
    probe = _probe_video(video)
    video_stream = next(stream for stream in probe["streams"] if stream.get("codec_type") == "video")
    actual_duration = float(probe["format"].get("duration", 0.0) or 0.0)
    motion = output_motion_score(video)
    receipt_path = WORK_ROOT / status["job_id"] / "composition_receipt.json"
    receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
    render_review = receipt.get("quality_review", {}).get("variants", {}).get("1", {}).get("render", {})
    report = {
        "label": label,
        "duration": actual_duration,
        "expected_duration": expected_duration,
        "width": int(video_stream.get("width", 0)),
        "height": int(video_stream.get("height", 0)),
        "motion_score": motion,
        "quality_ok": render_review.get("ok"),
        "render_features": render_review.get("checks", {}).get("render_features"),
        "receipt": str(receipt_path),
    }
    print(json.dumps(report, indent=2))
    assert abs(actual_duration - expected_duration) <= 0.75, report
    assert int(video_stream.get("width", 0)) == 1080, report
    assert int(video_stream.get("height", 0)) == 1920, report
    assert motion > 0.002, report
    assert render_review.get("ok") is True, render_review
    show_contact_sheet(video, label)
    return report


## 6. Smoke test A — deterministic FFmpeg fallback

This confirms source handling, transcription/caption plumbing, vertical
rendering, and the local-file download route without invoking the LLM or
network asset providers.


In [ ]:
deterministic_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": "Create a clean informative short with a strong hook and captions.",
    "preset": "clean_explainer",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": False,
    "refinement_iterations": 0,
    "sfx": {"mode": "off"},
}
deterministic_job = submit_job(deterministic_payload)
deterministic_status = poll_job(deterministic_job)
deterministic_video = download_variant(deterministic_status)
deterministic_archive = archive_job(deterministic_status, "deterministic")
show_video(deterministic_video)
print("Deterministic artifacts:", deterministic_archive)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 7. Smoke test B — one-pass local Transformers director

This is the first AI-directed pass. The model proposes a constrained JSON
composition plan; Pydantic validates it and the renderer, not the model,
owns all FFmpeg execution.


In [ ]:
local_director_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": "Make an informative, energetic vertical short. Keep the strongest moments, add a concise hook, and use readable captions.",
    "preset": "clean_explainer",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": True,
    "refinement_iterations": 0,
    "editorial_policy": {
        "content_type": "podcast",
        "source_audio": "preserve",
        "voiceover": "off",
        "supplementary_visuals": "off",
        "camera_motion": "punch_ins",
        "review_level": "strict",
    },
    "sfx": {"mode": "off"},
}
local_director_job = submit_job(local_director_payload)
local_director_status = poll_job(local_director_job, timeout_seconds=1800)
local_director_video = download_variant(local_director_status)
local_director_archive = archive_job(local_director_status, "local-director")
show_video(local_director_video)
print("Local-director artifacts:", local_director_archive)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 8. Smoke test C — full agentic draft/review/refinement loop

This runs the bounded orchestration phase: plan, render a draft, inspect
the draft receipt/quality signals, ask the local director for a correction,
and render the revised output. Malformed model JSON is repaired or the best
valid draft is retained according to the branch's refinement policy.


In [ ]:
agentic_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": "Turn this into a polished, highly watchable short. Open with a strong hook, preserve the most informative moments, use tasteful pacing and captions, and add supplementary layers only when they clearly improve understanding.",
    "preset": "viral_short",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": True,
    "refinement_iterations": 2,
    "refinement_strict": False,
    "editorial_policy": {
        "content_type": "podcast",
        "source_audio": "preserve",
        "voiceover": "off",
        "supplementary_visuals": "off",
        "camera_motion": "dynamic",
        "review_level": "strict",
    },
    "sfx": {"mode": "off"},
}
agentic_job = submit_job(agentic_payload)
agentic_status = poll_job(agentic_job, timeout_seconds=3600)
agentic_video = download_variant(agentic_status)
agentic_archive = archive_job(agentic_status, "agentic-refinement")
show_video(agentic_video)
print("Agentic artifacts:", agentic_archive)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 9A. Smoke test D2 — rendered graphics, camera motion, and transitions

This is the renderer-contract test that the earlier notebook did not have. It
uses the deterministic director so the test is not blocked by a small model's
JSON style, while the request still exercises the same policy and FFmpeg
compositor used by the AI-directed path. The receipt must show actual rendered
feature counts, not only planned metadata.


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the renderer-contract smoke test is about to run.
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


In [ ]:
render_program_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": (
        "Create a cinematic vertical short with a visible opening hook, one "
        "transcript-grounded bridge callout, motivated camera movement, and "
        "a tasteful dissolve between selected moments. Preserve the source "
        "speaker audio and never add a competing voiceover."
    ),
    "preset": "documentary",
    "target_seconds": 15,
    "output_count": 1,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": False,
    "refinement_iterations": 0,
    "editorial_policy": {
        "content_type": "documentary",
        "editing_intensity": "cinematic",
        "source_audio": "preserve",
        "voiceover": "off",
        "supplementary_visuals": "off",
        "motion_graphics": "hero",
        "camera_motion": "dynamic",
        "review_level": "strict",
    },
    "sfx": {"mode": "off"},
}
render_program_job = submit_job(render_program_payload)
render_program_status = poll_job(render_program_job, timeout_seconds=2400)
render_program_video = download_variant(render_program_status)
render_program_archive = archive_job(render_program_status, "render-program")
show_video(render_program_video)
print("Rendered-program artifacts:", render_program_archive)


In [ ]:
render_program_receipt = json.loads(
    (WORK_ROOT / render_program_status["job_id"] / "composition_receipt.json").read_text(
        encoding="utf-8"
    )
)
render_program_edit = render_program_receipt["edit_plan"]
render_program_review = render_program_receipt["quality_review"]
render_program_render = render_program_review["variants"]["1"]["render"]
rendered_features = render_program_render["checks"].get("render_features", {})

print(json.dumps({
    "planned_graphics": len(render_program_edit.get("graphics", [])),
    "planned_camera_tracks": len(render_program_edit.get("camera_tracks", [])),
    "planned_transitions": len(render_program_edit.get("transitions", [])),
    "rendered_features": rendered_features,
    "quality_ok": render_program_render.get("ok"),
}, indent=2))

assert render_program_edit["schema_version"] == "sh01/ai-edit@2"
assert render_program_render["ok"], render_program_render
actual = rendered_features.get("actual", {})
required = rendered_features.get("required", {})
for feature_name, required_count in required.items():
    assert actual.get(feature_name, 0) >= required_count, (feature_name, actual, required)


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the renderer-contract smoke test is about to run.
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 9B. Smoke test D3 — storyboard/contact-sheet and render QA gate

The catalog's strongest workflows inspect the actual render, not just the
plan.  This gate samples the finished MP4, verifies duration and vertical
dimensions, measures that the picture changes, checks the server's render
feature contract, and displays a contact sheet for human review.  It is a
mechanical/human review gate, not a claim that a small local model is a
reliable visual critic.


In [ ]:
render_program_review_report = review_rendered_output(
    render_program_status,
    render_program_video,
    "render-program",
    expected_duration=15.0,
)


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the rendered storyboard and quality gate is complete
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 9. Smoke test D — local supplementary media layers

If `AI_COMPOSE_LOCAL_ASSET_LIBRARY` points to a directory containing local
images/videos, this tests AI-selected picture-in-picture or partial-frame
layers. It is skipped rather than failing when no local library is mounted.


In [ ]:
local_asset_root = (
    os.environ.get("AI_COMPOSE_LOCAL_ASSET_LIBRARY", "").strip()
    or os.environ.get("AI_COMPOSE_ASSET_LIBRARY", "").strip()
)
if not local_asset_root:
    print("Skipped: AI_COMPOSE_LOCAL_ASSET_LIBRARY is not configured")
else:
    local_asset_path = Path(local_asset_root).expanduser()
    local_media = [
        p for p in local_asset_path.rglob("*")
        if p.is_file() and p.suffix.lower() in {".mp4", ".webm", ".mov", ".mkv", ".jpg", ".jpeg", ".png", ".webp"}
    ] if local_asset_path.exists() else []
    if not local_media:
        print("Skipped: local asset library is empty:", local_asset_path)
    else:
        local_assets_payload = {
            "unlimited_resources": True,
            "source_path": str(SOURCE),
            "brief": "Add one or two highly relevant supporting visuals as tasteful partial-frame overlays. Keep the speaker and captions readable, avoid decorative clutter, and justify each visual with the spoken topic.",
            "preset": "clean_explainer",
            "target_seconds": 15,
            "output_count": 1,
            "captions_enabled": True,
            "fetch_assets": True,
            "local_asset_library": str(local_asset_path),
            "asset_sources": {"local_library": True, "pexels": False, "pixabay": False, "unsplash": False, "freesound": False, "max_candidates_per_query": 6},
            "max_asset_layers": 2,
            "llm_required": True,
            "refinement_iterations": 1,
            "refinement_strict": False,
            "editorial_policy": {
                "content_type": "podcast",
                "source_audio": "preserve",
                "voiceover": "off",
                "supplementary_visuals": "require",
                "camera_motion": "punch_ins",
                "review_level": "strict",
            },
            "sfx": {"mode": "off"},
        }
        local_assets_job = submit_job(local_assets_payload)
        local_assets_status = poll_job(local_assets_job, timeout_seconds=3600)
        local_assets_video = download_variant(local_assets_status)
        local_assets_archive = archive_job(local_assets_status, "local-assets")
        show_video(local_assets_video)
        print("Local-asset artifacts:", local_assets_archive)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 10. Smoke test E — multi-variant output

This verifies that the same validated plan can produce multiple render
variants. It remains local and asset-free to isolate variant generation.


In [ ]:
variants_payload = {
    "unlimited_resources": True,
    "source_path": str(SOURCE),
    "brief": "Create a concise polished short with readable captions and strong pacing.",
    "preset": "viral_short",
    "target_seconds": 12,
    "output_count": 2,
    "captions_enabled": True,
    "fetch_assets": False,
    "llm_required": True,
    "refinement_iterations": 0,
    "editorial_policy": {
        "content_type": "podcast",
        "source_audio": "preserve",
        "voiceover": "off",
        "supplementary_visuals": "off",
        "camera_motion": "dynamic",
    },
    "sfx": {"mode": "off"},
}
variants_job = submit_job(variants_payload)
variants_status = poll_job(variants_job, timeout_seconds=2400)
variant_videos = [download_variant(variants_status, i) for i in (1, 2)]
variants_archive = archive_job(variants_status, "multi-variant")
for variant_video in variant_videos:
    show_video(variant_video)
print("Variant artifacts:", variants_archive)

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 11. Optional remote asset and adaptive-audio tests

These are disabled by default. Set the flags to `True` only after adding
the relevant provider values to the private `.env` and confirming that the
source/license policy is acceptable. The notebook never requires these
providers for the local workflow.


In [ ]:
RUN_REMOTE_ASSET_TEST = False
RUN_SFX_TEST = False

if RUN_REMOTE_ASSET_TEST:
    configured_remote_provider = any(
        os.environ.get(name, "").strip()
        for name in ("PEXELS_API_KEY", "PIXABAY_API_KEY", "UNSPLASH_ACCESS_KEY")
    )
    if not configured_remote_provider:
        print("Skipped remote asset test: no provider key is configured")
    else:
        remote_payload = {
            "unlimited_resources": True,
            "source_path": str(SOURCE),
            "brief": "Use only highly relevant supplementary visuals, with tasteful partial-frame placement and no distracting repetition.",
            "preset": "documentary",
            "target_seconds": 15,
            "output_count": 1,
            "captions_enabled": True,
            "fetch_assets": True,
            "asset_sources": {"local_library": False, "pexels": True, "pixabay": True, "unsplash": False, "freesound": False, "max_candidates_per_query": 6},
            "max_asset_layers": 2,
            "llm_required": True,
            "refinement_iterations": 1,
            "refinement_strict": False,
            "editorial_policy": {"supplementary_visuals": "prefer", "source_audio": "preserve", "voiceover": "off"},
            "sfx": {"mode": "off"},
        }
        remote_job = submit_job(remote_payload)
        remote_status = poll_job(remote_job, timeout_seconds=3600)
        remote_video = download_variant(remote_status)
        remote_archive = archive_job(remote_status, "remote-assets")
        show_video(remote_video)
        print("Remote-asset artifacts:", remote_archive)
else:
    print("Remote asset test disabled")

if RUN_SFX_TEST:
    sfx_root = (
        os.environ.get("AI_COMPOSE_LOCAL_SFX_LIBRARY", "").strip()
        or os.environ.get("AI_COMPOSE_SFX_LIBRARY", "").strip()
    )
    if not sfx_root or not Path(sfx_root).exists():
        print("Skipped SFX test: AI_COMPOSE_LOCAL_SFX_LIBRARY is not configured")
    else:
        sfx_payload = {
            "unlimited_resources": True,
            "source_path": str(SOURCE),
            "brief": "Create a polished short with restrained, context-aware transition sounds.",
            "preset": "viral_short",
            "target_seconds": 15,
            "output_count": 1,
            "captions_enabled": True,
            "fetch_assets": False,
            "local_sfx_library": sfx_root,
            "llm_required": True,
            "refinement_iterations": 0,
            "editorial_policy": {"sfx": "adaptive", "source_audio": "preserve", "voiceover": "off"},
            "sfx": {"mode": "adaptive", "local_library": sfx_root, "volume": 0.4},
        }
        sfx_job = submit_job(sfx_payload)
        sfx_status = poll_job(sfx_job, timeout_seconds=2400)
        sfx_video = download_variant(sfx_status)
        sfx_archive = archive_job(sfx_status, "adaptive-sfx")
        show_video(sfx_video)
        print("SFX artifacts:", sfx_archive)
else:
    print("SFX test disabled")

## 11A. Optional local voiceover and music branches

These branches are disabled by default because a talking source should not be
covered by a competing narrator.  Enable them only with a local Piper model
or a local/approved music URL in the private `.env`.  They exercise the
existing audio input path; beat-synchronous editing remains explicitly
future-backend work.


In [ ]:
RUN_VOICEOVER_TEST = False
RUN_MUSIC_TEST = False

if RUN_VOICEOVER_TEST:
    tts_model = os.environ.get("AI_COMPOSE_TTS_MODEL", "").strip()
    if not tts_model:
        print("Skipped voiceover test: AI_COMPOSE_TTS_MODEL is not configured")
    else:
        voiceover_payload = {
            "unlimited_resources": True,
            "source_path": str(SOURCE),
            "brief": "Add a short hook voiceover only if it does not compete with the source narrative.",
            "preset": "viral_short",
            "target_seconds": 15,
            "output_count": 1,
            "captions_enabled": True,
            "fetch_assets": False,
            "llm_required": False,
            "refinement_iterations": 0,
            "voiceover": {"mode": "hook_only", "text": "Here is the key idea.", "voice": tts_model, "volume": 0.55},
            "editorial_policy": {"source_audio": "preserve", "voiceover": "hook_only", "supplementary_visuals": "off"},
            "sfx": {"mode": "off"},
        }
        voiceover_status = poll_job(submit_job(voiceover_payload), timeout_seconds=2400)
        voiceover_video = download_variant(voiceover_status)
        archive_job(voiceover_status, "local-voiceover")
        show_video(voiceover_video)
else:
    print("Voiceover test disabled")

if RUN_MUSIC_TEST:
    music_url = os.environ.get("AI_COMPOSE_MUSIC_URL", "").strip()
    if not music_url:
        print("Skipped music test: AI_COMPOSE_MUSIC_URL is not configured")
    else:
        music_payload = {
            "unlimited_resources": True,
            "source_path": str(SOURCE),
            "brief": "Add restrained background music beneath the source narrative.",
            "preset": "documentary",
            "target_seconds": 15,
            "output_count": 1,
            "captions_enabled": True,
            "fetch_assets": False,
            "llm_required": False,
            "refinement_iterations": 0,
            "music_url": music_url,
            "music_volume": 0.08,
            "editorial_policy": {"source_audio": "preserve", "voiceover": "off", "music": "library", "supplementary_visuals": "off"},
            "sfx": {"mode": "off"},
        }
        music_status = poll_job(submit_job(music_payload), timeout_seconds=2400)
        music_video = download_variant(music_status)
        archive_job(music_status, "music-mix")
        show_video(music_video)
else:
    print("Music test disabled")


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the optional audio branch is complete
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 12. Optional long-source test

For a large/long input, set `RUN_LONG_SOURCE_TEST=True`. The server should
chunk analysis and render only the requested target duration while keeping
memory bounded. Expect this test to take substantially longer than the
15-second smoke tests.


In [ ]:
RUN_LONG_SOURCE_TEST = False
if RUN_LONG_SOURCE_TEST:
    long_payload = {
        "unlimited_resources": True,
        "source_path": str(SOURCE),
        "brief": "Find the strongest narrative moments in this long source and make a coherent informative short with captions.",
        "preset": "documentary",
        "target_seconds": 45,
        "output_count": 1,
        "captions_enabled": True,
        "fetch_assets": False,
        "llm_required": True,
        "refinement_iterations": 1,
        "refinement_strict": False,
        "editorial_policy": {"source_audio": "preserve", "voiceover": "off", "supplementary_visuals": "off", "review_level": "strict"},
        "sfx": {"mode": "off"},
    }
    long_job = submit_job(long_payload)
    long_status = poll_job(long_job, timeout_seconds=7200, interval_seconds=10)
    long_video = download_variant(long_status)
    long_archive = archive_job(long_status, "long-source")
    show_video(long_video)
    print("Long-source artifacts:", long_archive)
else:
    print("Long-source test disabled")

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


## 13A. Final capability report

In [ ]:
print("\nNotebook capability summary:")
for row in CATALOG_CAPABILITY_MATRIX:
    print(f"- {row['status']:<19} {row['family']}  [{row['test']}]")

print("\nThe following remain intentionally unclaimed by this FFmpeg-first server:")
for row in CATALOG_CAPABILITY_MATRIX:
    if row["status"] in {"future-backend", "partial"}:
        print(f"- {row['status']}: {row['family']}")


In [ ]:
from IPython.display import Audio, display
import numpy as np

# Audible checkpoint: the comprehensive capability report is complete
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))


## 13. Receipts, logs, and clean shutdown

Each completed job receipt includes the selected source range, word/frame
counts, asset requests/resolution, refinement metadata, and output files.
The log tail is useful if a provider or model fails. Shut down the local
server after reviewing the outputs.


In [ ]:
print("Completed receipts:", len(JOB_RECEIPTS))
for receipt in JOB_RECEIPTS:
    print(json.dumps({
        "job_id": receipt.get("job_id"),
        "status": receipt.get("status"),
        "source_duration": receipt.get("source_duration"),
        "word_count": receipt.get("word_count"),
        "asset_queries": receipt.get("asset_queries"),
        "resolved_assets": receipt.get("resolved_assets"),
        "refinement_round": receipt.get("refinement_round"),
        "outputs": receipt.get("outputs"),
        "warnings": receipt.get("warnings"),
    }, indent=2))

print("Archived output folders:")
for path in sorted(OUTPUT_ROOT.iterdir()):
    if path.is_dir():
        print(" -", path)

if "SERVER_PROCESS" in globals() and SERVER_PROCESS.poll() is None:
    SERVER_PROCESS.terminate()
    try:
        SERVER_PROCESS.wait(timeout=20)
    except subprocess.TimeoutExpired:
        SERVER_PROCESS.kill()
        SERVER_PROCESS.wait(timeout=10)
if "SERVER_LOG_HANDLE" in globals():
    SERVER_LOG_HANDLE.close()
print("Server stopped")
print("Log:", SERVER_LOG)
print(SERVER_LOG.read_text(encoding="utf-8", errors="replace")[-5000:])

In [ ]:
from IPython.display import Audio, display
import numpy as np

# Generate a short 440 Hz sine wave beep
sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)

display(Audio(wave, rate=sample_rate, autoplay=True))


In [ ]:
for receipt in JOB_RECEIPTS:
    path = WORK_ROOT / receipt["job_id"] / "composition_receipt.json"
    if not path.exists():
        continue

    data = json.loads(path.read_text(encoding="utf-8"))
    plan = data["plan"]
    print("\nJOB:", receipt["job_id"])
    print("Base segments:", plan["base_segments"])
    print("Asset requests:", plan["asset_requests"])
    print("Resolved layers:", len(data["resolved_layers"]))
    print("SFX events:", plan["sfx_events"])
    print("Voiceovers:", plan["voiceovers"])
    print("Editorial profile:", data.get("editorial_profile", {}))
    print("Edit-plan version:", data.get("edit_plan", {}).get("schema_version"))
    print("Quality review:", data.get("quality_review", {}))

## 14. Package all artifacts for download

Every completed job has been copied into `/kaggle/working/ai-compose-output`.
The zip is created only after the server has stopped, so it includes the
rendered MP4s, receipts, evidence, ASS captions, and status JSON files.

In [ ]:
archive_base = "/kaggle/working/sh01-ai-compose-results"
archive_path = Path(shutil.make_archive(archive_base, "zip", OUTPUT_ROOT))
print("Download archive:", archive_path)
print("Archive size MiB:", round(archive_path.stat().st_size / 1024**2, 2))

from IPython.display import Audio, display
import numpy as np

sample_rate = 8000
t = np.linspace(0, 0.5, int(sample_rate * 0.5), endpoint=False)
wave = np.sin(2 * np.pi * 440 * t)
display(Audio(wave, rate=sample_rate, autoplay=True))